In [ ]:
# @title Basic Scrapping Code
import requests
from bs4 import BeautifulSoup
import time
import csv

# We scrape across multiple topical sections to gather a robust baseline pool
TARGET_SECTIONS = [
    "https://www.bernama.com/en/",
    "https://www.bernama.com/en/general/",
    "https://www.bernama.com/en/business/"
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

def harvest_article_links(section_url):
    """
    Step 1: Scans the target section landing page and collects unique article URLs.
    """
    print(f"[*] Scanning section: {section_url}")
    links_pool = []

    try:
        response = requests.get(section_url, headers=HEADERS, timeout=15)
        if response.status_code != 200:
            print(f"[-] Failed to access section: HTTP {response.status_code}")
            return []

        soup = BeautifulSoup(response.text, 'html.parser')
        anchors = soup.find_all('a', href=True)

        for anchor in anchors:
            href = anchor['href']

            # Exact match for the pattern seen in image_4dc6bd.png
            if "news.php?id=" in href:
                # Build an absolute URL format cleanly
                full_url = href if href.startswith("http") else f"https://www.bernama.com/en/{href}"

                # Strip out excess tracking elements if attached
                full_url = full_url.split('&')[0] if '&' in full_url else full_url

                if full_url not in links_pool:
                    links_pool.append(full_url)

        return links_pool

    except Exception as e:
        print(f"[-] Error parsing section index: {e}")
        return []


def parse_bernama_article(article_url):
    """
    Step 2: Visits individual article URLs and extracts targeted data fields.
    """
    try:
        response = requests.get(article_url, headers=HEADERS, timeout=15)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, 'html.parser')

        # 1. Headline (Matches <h1> tag with class 'fw-bold h2' from image_4dbff6.png)
        headline_tag = soup.find('h1', class_='fw-bold')
        if not headline_tag:
            headline_tag = soup.find('h1') # Fallback
        headline = headline_tag.get_text(strip=True) if headline_tag else None

        # 2. Publication Date (Matches <div class='text-start'> containing time from image_4dbf5b.png)
        date_div = soup.find('div', class_='text-start')
        pub_date = date_div.get_text(strip=True) if date_div else "N/A"

        # 3. Section Classification
        section = "News"
        if "business" in article_url:
            section = "Business"
        elif "general" in article_url:
            section = "General"

        # 4. Article Summary (Extracting the first clean paragraph of copy)
        summary = None
        # Look inside the primary article text container
        paragraphs = soup.find_all('p')
        for p in paragraphs:
            text = p.get_text(strip=True)
            # Filter out short metadata strings or social link headers
            if len(text) > 40:
                summary = text
                break

        if headline:
            return {
                "headline": headline,
                "publication_date": pub_date,
                "section": section,
                "article_summary": summary,
                "url": article_url
            }
        return None

    except Exception:
        return None


if __name__ == "__main__":
    start_time = time.time()
    all_records = []

    csv_file = "bernama_baseline_raw.csv"
    fields = ["headline", "publication_date", "section", "article_summary", "url"]

    print("[*] Launching BERNAMA Resilient Baseline Pipeline...")

    # Harvest links across all sections first
    discovered_links = []
    for section in TARGET_SECTIONS:
        section_links = harvest_article_links(section)
        discovered_links.extend(section_links)
        time.sleep(1) # Polite pause between indices

    # Remove duplicates
    unique_links = list(set(discovered_links))
    print(f"\n[+] Total unique deep article links harvested: {len(unique_links)}")

    # Process and progressively save deep contents
    if unique_links:
        print(f"[*] Beginning deep extraction into '{csv_file}'...")
        with open(csv_file, mode="w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=fields)
            writer.writeheader()

            for index, link in enumerate(unique_links, 1):
                article_id = link.split('=')[-1]
                print(f"    [{index}/{len(unique_links)}] Processing Article ID: {article_id}...")

                article_data = parse_bernama_article(link)
                if article_data:
                    all_records.append(article_data)
                    writer.writerow(article_data) # Save row directly to disk

                time.sleep(1) # Ethical rate-limiting step

    end_time = time.time()
    print("\n" + "="*50)
    print(f"[SUCCESS] Scraped {len(all_records)} valid records successfully!")
    print(f"[METRIC] Total Execution Time: {end_time - start_time:.2f} seconds.")
    print(f"[DATA] Saved copy located at: '{csv_file}'")
    print("="*50)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import csv
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from dateutil import parser as dateparser

# We scrape across multiple topical sections to gather a robust baseline pool
# Replace section URLs with ID range for bulk harvesting
ID_START = 1780000
ID_END =  1790000

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

# ADD after HEADERS dict:
session = requests.Session()
retries = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 503])
session.mount("https://", HTTPAdapter(max_retries=retries))

def generate_article_links(id_start, id_end):
    """
    Step 1: Generates article URLs from a sequential ID range.
    No need to scrape section pages — IDs are predictable.
    """
    print(f"[*] Generating article URLs from ID {id_start} to {id_end}...")
    links_pool = [
        (f"https://www.bernama.com/en/news.php?id={article_id}", "News")
        for article_id in range(id_start, id_end + 1)
    ]
    print(f"[+] Total URLs generated: {len(links_pool)}")
    return links_pool


def parse_bernama_article(article_url, section="News"):
    """
    Step 2: Visits individual article URLs and extracts targeted data fields.
    """
    try:
        response = session.get(article_url, headers=HEADERS, timeout=15)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, 'html.parser')

        # 1. Headline (Matches <h1> tag with class 'fw-bold h2' from image_4dbff6.png)
        headline_tag = soup.find('h1', class_='fw-bold')
        if not headline_tag:
            headline_tag = soup.find('h1') # Fallback
        headline = headline_tag.get_text(strip=True) if headline_tag else None

        # 2. Publication Date (Matches <div class='text-start'> containing time from image_4dbf5b.png)
        date_div = soup.find('div', class_='text-start')
        raw_date = date_div.get_text(strip=True) if date_div else None
        try:
            pub_date = dateparser.parse(raw_date).strftime("%Y-%m-%d %H:%M") if raw_date else "N/A"
        except Exception:
            pub_date = raw_date or "N/A"

        # 4. Article Summary (Extracting the first clean paragraph of copy)
        summary = None
        # Look inside the primary article text container
        # Scope to article body first to avoid nav/footer text
        article_body = soup.find('div', class_='article-body')
        paragraphs = article_body.find_all('p') if article_body else soup.find_all('p')
        for p in paragraphs:
            text = p.get_text(strip=True)
            if len(text) > 40:
                summary = text
                break

        if headline:
            return {
                "headline": headline,
                "publication_date": pub_date,
                "section": section,
                "article_summary": summary,
                "url": article_url
            }
        return None

    except Exception:
        return None


if __name__ == "__main__":
    start_time = time.time()
    all_records = []

    csv_file = "/content/drive/MyDrive/bernama_baseline_raw.csv"
    fields = ["headline", "publication_date", "section", "article_summary", "url"]

    print("[*] Launching BERNAMA Resilient Baseline Pipeline...")

    # EDIT: replaced section scraping with ID range generation
    unique_links = generate_article_links(ID_START, ID_END)
    print(f"\n[+] Total article URLs queued: {len(unique_links)}")

    # checkpointing — skip already scraped URLs on re-run
    scraped_urls = set()
    file_mode = "w"
    try:
        with open(csv_file, encoding="utf-8") as f:
            scraped_urls = {row['url'] for row in csv.DictReader(f)}
        file_mode = "a"
        print(f"[*] Resuming — {len(scraped_urls)} articles already scraped, skipping.")
    except FileNotFoundError:
        file_mode = "w"
        print("[*] No existing file found — starting fresh.")

    unique_links = [(url, label) for url, label in unique_links if url not in scraped_urls]

    # this line is to test with only first 5 articles — DELETE this line when running full scrape
    # unique_links = unique_links[:1000]

    if unique_links:
        print(f"[*] Beginning deep extraction into '{csv_file}'...")
        with open(csv_file, mode=file_mode, newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=fields)
            if file_mode == "w":
                writer.writeheader()

            # concurrent scraping replacing sequential for-loop
            with ThreadPoolExecutor(max_workers=5) as executor:
                futures = {
                    executor.submit(parse_bernama_article, url, label): url
                    for url, label in unique_links
                }
                completed = 0
                for future in as_completed(futures):
                    completed += 1
                    result = future.result()
                    if result:
                        all_records.append(result)
                        writer.writerow(result)
                        print(f"    [{completed}/{len(unique_links)}] Scraped: {result['headline'][:60]}...")

    end_time = time.time()
    print("\n" + "="*50)
    print(f"[SUCCESS] Scraped {len(all_records)} valid records successfully!")
    print(f"[METRIC] Total Execution Time: {end_time - start_time:.2f} seconds.")
    print(f"[DATA] Saved copy located at: '{csv_file}'")
    print("="*50)

[*] Launching BERNAMA Resilient Baseline Pipeline...
[*] Generating article URLs from ID 1780000 to 1790000...
[+] Total URLs generated: 10001

[+] Total article URLs queued: 10001
[*] Resuming — 97351 articles already scraped, skipping.
[*] Beginning deep extraction into '/content/drive/MyDrive/bernama_baseline_raw.csv'...
    [3/10000] Scraped: Go into vegetable, fruit farming - Dr Mahathir...
    [6/10000] Scraped: KLIBOR futures untraded at mid-day...
    [9/10000] Scraped: Singapore, US sign deal on infrastructure finance, market bu...
    [11/10000] Scraped: BIMB Investment expects 10-15 per cent growth for Shariah-ES...
    [20/10000] Scraped: MED discusses direction of air mobility industry with CAAM, ...
    [21/10000] Scraped: Functions of cooperative institutions need to be strengthene...
    [26/10000] Scraped: Labuan police intensify crime prevention efforts...
    [27/10000] Scraped: Sweden to raise retirement age...
    [29/10000] Scraped: Lithium Werks prioritises water